### Assistant

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [39]:
#tools

def add(a:int , b:int)->str:
    """Basic tool for addition of intergers"""
    return f"The sum of numbers is {a+b}"

def subtration(a:int , b:int)->str:
    """Basic tool for subtraction of intergers"""
    return f"The diffrence of numbers is {a-b}"

def multiplication(a:int , b:int)->str:
    """Basic tool for multiplication of intergers"""
    return f"The product of numbers is {a*b}"

def division(a:int , b:int)->str:
    """Basic tool for division of intergers"""
    if b==0: 
        return "cannot divide by zero"
    return f"The division of numbers is {a/b}"
def power(base:int , exponent:int)->int:
    """
    This a tool to calcuate the power of a number 
    use it whenever users asks for power of a number
       
    """
    return base**exponent

#Unit converter

def distance_converter(value:float,from_unit:str,to_unit:str)->str:
    """

Convert distance between units.

Parameters:

value:
A numeric distance value.
Must be a number.

from_unit:
Source unit.

to_unit:
Target unit.
    """
    from_unit=from_unit.lower()
    to_unit=to_unit.lower()

    aliases = {
        "km": "km",
        "kms": "km",
        "kilometer": "km",
        "kilometers": "km",
        "kilometre": "km",
        "kilometres": "km",

        "mile": "mile",
        "miles": "mile",

        "meter": "meter",
        "meters": "meter",
        "metre": "meter",
        "metres": "meter",

        "foot": "foot",
        "feet": "foot",
        "ft": "foot"
    }
        
    from_unit = aliases.get(from_unit)
    to_unit = aliases.get(to_unit)
    
    if from_unit=="km" and to_unit=="mile":
        return f"The result in miles is {value*0.621371}"
    elif from_unit=="mile" and to_unit=="km":
        return f"The result in km is {value*1.60934}"
    elif from_unit=="meter" and to_unit=="foot":
        return f"The result in feet is {value*3.28084}"
    elif from_unit=="foot" and to_unit=="meter":
        return f"The result in meters is {value*0.3048}"
    else:
        return "Conversion not supported"

def weight_converter(W:float,from_unit:str,to_unit:str)->str:
    """
    This tool is used to convert weight in units.
    Use this whenever user asks for converting weight.
    You can convert from kg to pounds and viceversa.
    """

    from_unit=from_unit.lower()
    to_unit=to_unit.lower()

    aliases = {
        "kg": "kg",
        "kgs": "kg",
        "kilogram": "kg",
        "kilograms": "kg",

        "lb": "lb",
        "lbs": "lb",
        "pound": "lb",
        "pounds": "lb"
    }
    from_unit = aliases.get(from_unit)
    to_unit = aliases.get(to_unit)

    if from_unit=="kg" and to_unit=="lb":
        return f"The result in pounds is {W*2.20462}"
    elif from_unit=="lb" and to_unit=="kg":
        return f"The result in kilograms is {W*0.453592}"
    else:
        return "Conversion not supported"

def temperature_converter(T:float,from_unit:str,to_unit:str)->str:
    """
    This tool is used to convert temperature in units.
    Use this whenever user asks for converting temperature.
    You can convert from Fahrenheit to celsius and viceversa.
    """
    from_unit=from_unit.lower()
    to_unit=to_unit.lower()
    
    aliases = {
        "c": "c",
        "celsius": "c",

        "f": "f",
        "fahrenheit": "f"
    }

    from_unit = aliases.get(from_unit)
    to_unit = aliases.get(to_unit)

    if from_unit=="f" and to_unit=="c":
        return f"The result in celsius is {(T-32)*(5/9)}"
    elif from_unit=="c" and to_unit=="f":
        return f"The result in Fahrenheit is {(T*1.8)+32}"
    else:
        return "Conversion not supported"


#Contact Extractor

from collections import defaultdict

database = defaultdict(dict)


def save_contact(name:str , email:str , phone:str)->str:
    """
    This tool is used to save contact in database.
    Use it whenever users asks you to save contact or 
    save details with name, email and phone number given.
    """
    name=name.lower()
    database[name]["email"]=email
    database[name]["phone"]=phone
   # print(dict(database))
    return "The data has been saved in database"

def find_contact(name:str)->str:
    """ 
    This tool is used to find contact in database.
    Use it whenever user asks you to find a particular contact
    with the name given.
    """
    name=name.lower()
    contact = database.get(name)
    if not contact:
        return f"The Contact {name} not found."
    return f"The contact details of {name} are {database[name]['email']}"

def delete_contact(name:str)->str:
    """
    This tool is used to delete contact in databse.
    Use it when user asks you to delete a particlar contact
    with the given name.
    """
    name=name.lower()
    removed_contact = database.pop(name, None)
    if not removed_contact:
        return f"Contact {name} not found."
    return f"Contact {name} has been successfully deleted."

def list_contacts():
    """
    This tool is used to save contacts in database.
    Use it when the user asks you to list all the saved contacts
    or just print all databse items.
    """
    if not database:
        print("database is empty.")
        return
    for name,details in database.items():
        email=details.get('email','N/A')
        phone=details.get('phone','N/A')
        return f"The contact for {name} : email {email} , phone{phone}"  

def read_email_tool(email_id:str)->str:
    """mock function to read email"""
    return f"email content for ID:{email_id}"


def send_email_tool(recipient:str , subject:str, body:str)->str:
    """function to send email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [23]:
config={"configurable":{"thread_id":"test-approve"}}

In [43]:
from langchain.agents import create_agent
from pydantic import BaseModel,Field
from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage,AIMessage,SystemMessage
class contract(BaseModel):
    """ Basic details of a contract"""
    name:str=Field(description="The name of the user")
    email:str=Field(description="The email of the user")
    phone:str=Field(description="The phone number of the user")


system_prompt="""
        You are Nischall's Personal AI Assistant.

        Responsibilities:
        - Solve mathematical problems using calculator tools.
        - Convert units using conversion tools.
        - Manage contacts using contact tools.
        - Never guess contact information.
        - Always use available tools instead of reasoning mentally.
        - If an email needs to be sent, wait for human approval.
        - Keep responses concise and professional.
"""

Assistant=create_agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt=system_prompt,
    tools=[add,subtration,multiplication,division,distance_converter,weight_converter,temperature_converter,save_contact,find_contact,delete_contact,list_contacts],
    # response_format=contract,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("messages",10),
            keep=("messages",4)
        ),
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decision":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)
# for chunk in model.stream("what is water?"):
#     print(chunk.text,end="|",flush=True)
questions=[
    #"what is the sum of 2 and 4",
    # "what is the procut of 25*5",
    # "convet 35 kilograms into pounds",
    # "convert 5 degree celsius into fahrenheit",
    #"save the contact with name rohit, email 1234rocky@gmail.com , 233-1232-333",
    # "save the contact with name priya, email 1234pony@gmail.com , 233-2323-122",
    # "Find the contact with name priya",
    # "delete the contact with name rohit",
    # "list all the contacts",
    #"send an email to john@123.in with subject hello , body nice to meet you",

]
# Human message
# response=Assistant.invoke(
#         {
#             "messages":[
#                 HumanMessage(
#                     content="convert 20 miles to km",
#                 )
#             ]
#         },
#         config=config,
#     )
# response

for chunk in Assistant.stream(
    {
        "messages":[
            HumanMessage(
                content="Explain LangChain."
            )
        ]
    },
    config=config
):
    print(chunk)

# for q in questions:
#     response=Assistant.invoke(
#         {
#             "messages":[
#                 {
#                     "role":"user",
#                     "content":q
#                 },
#             ]
#         },
#         config=config,
#     )
#     print("="*50)
#     print(f"The question {q}")
    
#     for msg in response["messages"]:
#         print(type(msg).__name__)
#         print(msg)
#         print("-" * 40)

{'SummarizationMiddleware.before_model': None}
{'model': {'messages': [AIMessage(content='LangChain is an open-source framework designed to help developers build applications that utilize large language models (LLMs) more efficiently. It provides a set of tools and libraries that simplify the process of integrating LLMs into various projects, allowing developers to focus on building their applications rather than worrying about the underlying complexities of the language models.\n\nSome key features of LangChain include:\n\n1. **Modular architecture**: LangChain allows developers to break down their application into smaller, independent components, making it easier to manage and maintain their codebase.\n2. **Agent-based framework**: LangChain introduces the concept of "agents," which are autonomous entities that can interact with LLMs to perform specific tasks. This framework enables developers to build more complex and dynamic applications.\n3. **Memory and knowledge management**: La